# Phase 1 Robustness Checks

Supplementary analyses validating the main DiD and Causal Forest results.

In [1]:
!pip install pyfixest

  Using cached formulaic-1.2.1-py3-none-any.whl.metadata (7.0 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached maketables-0.1.8-py3-none-any.whl.metadata (9.2 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached interface_meta-1.3.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached great_tables-0.21.0-py3-none-any.whl.metadata (13 kB)
  Using cached python_docx-1.2.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached commonmark-0.9.1-py2.py3-none-any.whl.metadata (5.7 kB)
  Using cached faicons-0.2.2-py3-none-any.whl.metadata (1.8 kB)
  Using cached htmltools-0.6.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached babel-2.18.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached importlib_resources-7.1.0-py3-none-any.whl.metadata (4.0 kB)
   ---------------------------------------- 0.0/3.1 MB ? eta -:--:--
   ------ --------------------------------- 0.5/3.1 MB 16.2 MB/s eta 0:00:01
   ------------- -------------------------- 1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spyder 5.4.3 requires ipython!=8.10.0,!=8.8.0,!=8.9.0,<9.0.0,>=7.31.1, but you have ipython 9.3.0 which is incompatible.
spyder 5.4.3 requires jedi<0.19.0,>=0.17.2, but you have jedi 0.19.2 which is incompatible.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/cleaned/final_analysis_data.csv')

print(f"Shape: {df.shape}")
print(f"Treated obs (post_carbon_tax): {df['post_carbon_tax'].sum()}")
print(f"Treatment cohorts: {sorted(df.loc[df['treatment_year'].notna(), 'treatment_year'].unique().astype(int))}")
print(f"Never-treated countries: {df[df['treatment_year'].isna()]['country'].nunique()}")

Shape: (3892, 69)
Treated obs (post_carbon_tax): 511
Treatment cohorts: [2000, 2005, 2008, 2010, 2012, 2013, 2015, 2016, 2017, 2018, 2019, 2020, 2022]
Never-treated countries: 122


## 1. Staggered-Robust DiD (Gardner DID2S)

Standard TWFE with 13 treatment cohorts is vulnerable to negative weight bias when treatment effects are heterogeneous (Goodman-Bacon 2021). Gardner (2022) DID2S is a two-stage estimator that avoids this bias. We compare it against standard TWFE.

In [ ]:
import pyfixest as pf

# Prepare data for pyfixest event_study()
df_staggered = df.copy()

# Never-treated: cohort = 0 (pyfixest convention)
df_staggered['cohort'] = df_staggered['treatment_year'].fillna(0).astype(int)

# Encode country as integer
df_staggered['country_id'] = df_staggered['country'].astype('category').cat.codes

outcome = 'co2_per_capita_future_trend'
if outcome not in df_staggered.columns or df_staggered[outcome].isna().all():
    outcome = 'co2_per_capita_3yr_change'

df_staggered = df_staggered[df_staggered[outcome].notna()].copy()

print(f"Staggered sample: {df_staggered.shape}")
print(f"Cohorts: {sorted(df_staggered['cohort'].unique())}")
print(f"Outcome variable: {outcome}")

In [ ]:
# Gardner DID2S — staggered-robust, no negative weights
# Stage 1: estimate FE using only untreated/not-yet-treated obs
# Stage 2: estimate treatment effects on residualized outcome
fit_did2s = pf.event_study(
    data=df_staggered,
    yname=outcome,
    idname='country_id',
    tname='year',
    gname='cohort',
    estimator='did2s',
    cluster='country_id',
    att=True
)

# Standard TWFE for comparison
fit_twfe = pf.event_study(
    data=df_staggered,
    yname=outcome,
    idname='country_id',
    tname='year',
    gname='cohort',
    estimator='twfe',
    cluster='country_id',
    att=True
)

print("=== DID2S (Gardner 2022, staggered-robust) ===")
print(fit_did2s.summary())
print("\n=== TWFE (standard, for comparison) ===")
print(fit_twfe.summary())

In [ ]:
import os
os.makedirs('../outputs', exist_ok=True)

# Event study plot — DID2S vs TWFE
pf.iplot([fit_did2s, fit_twfe], figsize=(12, 6),
         title="Staggered-Robust Event Study: DID2S vs TWFE")
plt.savefig('../outputs/did2s_event_study.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to outputs/did2s_event_study.png")

In [ ]:
# Extract ATT estimates for comparison table
did2s_coefs = fit_did2s.coef()
twfe_coefs = fit_twfe.coef()

# Post-treatment ATTs (positive relative time periods)
did2s_post = {k: v for k, v in did2s_coefs.items()}
twfe_post = {k: v for k, v in twfe_coefs.items()}

did2s_att = np.mean(list(did2s_post.values())) if did2s_post else np.nan
twfe_att = np.mean(list(twfe_post.values())) if twfe_post else np.nan

print(f"=== Average Post-Treatment Effect ===")
print(f"  DID2S (staggered-robust): {did2s_att:.4f}")
print(f"  TWFE  (standard):         {twfe_att:.4f}")
print(f"  Difference:               {did2s_att - twfe_att:.4f}")
if abs(twfe_att) > 0:
    print(f"  TWFE bias (approx):       {((twfe_att - did2s_att)/did2s_att)*100:.1f}%")

## Staggered DiD Robustness Check

| Estimator | ATT Estimate | Notes |
|-----------|-------------|-------|
| TWFE (standard) | fill in | Susceptible to negative weights with 13 cohorts |
| DID2S (Gardner 2022) | fill in | Staggered-robust, two-stage estimator |

**Conclusion:** fill in after running

## 2. ETS Control Group Sensitivity

EU ETS countries (carbon pricing via cap-and-trade but no carbon tax) are in the control group. This contaminates "untreated" status and may bias the treatment effect downward. Below we test three control group definitions.

In [ ]:
# Identify ETS countries in control group
print("ETS column check:")
print(df['has_ets'].value_counts())
print(f"\nETS countries in control group (has_ets=1, post_carbon_tax=0):")
ets_control = df[(df['has_ets'] == 1) & (df['post_carbon_tax'] == 0)]['country'].unique()
print(sorted(ets_control))
print(f"\nCount: {len(ets_control)} countries")

In [ ]:
import statsmodels.formula.api as smf

DID_FORMULA = """
co2_per_capita_future_trend ~ post_carbon_tax
    + C(year) + C(country)
    + log_gdp + log_population + trade_openness
    + natural_resource_rents_per_gdp + fossil_pct_filled
"""

def run_did(data, label):
    """Run DiD with clustered SEs and return results dict."""
    model = smf.ols(DID_FORMULA, data=data.dropna(
        subset=['co2_per_capita_future_trend', 'post_carbon_tax', 'log_gdp',
                'log_population', 'trade_openness', 'natural_resource_rents_per_gdp',
                'fossil_pct_filled']
    )).fit(
        cov_type='cluster', cov_kwds={'groups': data.dropna(
            subset=['co2_per_capita_future_trend', 'post_carbon_tax', 'log_gdp',
                    'log_population', 'trade_openness', 'natural_resource_rents_per_gdp',
                    'fossil_pct_filled']
        )['country']}
    )
    coef = model.params['post_carbon_tax']
    se = model.bse['post_carbon_tax']
    pval = model.pvalues['post_carbon_tax']
    n_countries = data['country'].nunique()
    n_obs = int(model.nobs)
    print(f"  {label:<45} coef={coef:7.4f}  SE={se:.4f}  p={pval:.3f}  "
          f"N={n_obs}  countries={n_countries}")
    return {'label': label, 'coef': coef, 'se': se, 'pval': pval, 'n': n_obs}

# Spec A: original (ETS countries in control)
df_spec_a = df.copy()

# Spec B: exclude ETS-only countries from control group
# Keep: treated (carbon tax) + pure never-treated (no tax, no ETS)
df_spec_b = df[~((df['has_ets'] == 1) & (df['post_carbon_tax'] == 0))].copy()

# Spec C: only never-treated controls (no carbon tax AND no ETS ever)
never_any_pricing = df.groupby('country').apply(
    lambda x: (x['has_ets'].max() == 0) and (x['post_carbon_tax'].max() == 0)
)
never_any_countries = never_any_pricing[never_any_pricing].index.tolist()
treated_countries = df[df['post_carbon_tax'] == 1]['country'].unique().tolist()
df_spec_c = df[df['country'].isin(never_any_countries + treated_countries)].copy()

print("=== ETS Sensitivity Analysis ===\n")
results_a = run_did(df_spec_a, "A: Original (ETS in control)")
results_b = run_did(df_spec_b, "B: Exclude ETS-only countries")
results_c = run_did(df_spec_c, "C: Never-any-pricing controls only")

In [ ]:
results = [results_a, results_b, results_c]
labels = [r['label'] for r in results]
coefs = [r['coef'] for r in results]
ses = [r['se'] for r in results]

fig, ax = plt.subplots(figsize=(9, 5))
y_pos = np.arange(len(results))

ax.errorbar(coefs, y_pos, xerr=[1.96 * s for s in ses],
            fmt='o', color='steelblue', capsize=5, markersize=8, linewidth=2)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel('DiD Coefficient (Effect on CO₂/Capita Trend)', fontsize=11)
ax.set_title('ETS Control Group Sensitivity\n(All specs cluster SEs at country level)', fontsize=12)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/ets_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to outputs/ets_sensitivity.png")

## ETS Control Group Sensitivity

| Spec | Description | Coef | SE | p-value |
|------|-------------|------|----|---------|
| A | Original (ETS in control) | fill in | fill in | fill in |
| B | Exclude ETS-only countries | fill in | fill in | fill in |
| C | Never-any-pricing controls only | fill in | fill in | fill in |

**Finding:** fill in after running

**Preferred specification for Paper 1:** fill in

## 3. Effective Carbon Price (Tax Price x Coverage Rate)

`tax_price` alone is insignificant (p=0.366) because it ignores sector coverage. Sweden ($130/tonne, ~40% coverage) vs Mexico ($3/tonne, ~47% coverage) have very different effective prices. This section constructs `effective_carbon_price = tax_price × coverage_pct`.

**Data source:** Download coverage rates from World Bank Carbon Pricing Dashboard (https://carbonpricingdashboard.worldbank.org/) or OECD Effective Carbon Rates. Save as `data/raw/carbon_tax_coverage.csv` with columns: `country`, `year` (optional), `coverage_pct` (0-100 scale).

In [ ]:
# Load coverage data — adjust path/columns after downloading
import os

coverage_path = '../data/raw/carbon_tax_coverage.csv'
if not os.path.exists(coverage_path):
    print(f"ERROR: {coverage_path} not found.")
    print("Download coverage rate data from:")
    print("  - World Bank Carbon Pricing Dashboard: https://carbonpricingdashboard.worldbank.org/")
    print("  - OR OECD Effective Carbon Rates: https://stats.oecd.org/Index.aspx?DataSetCode=ECR")
    print("Save as data/raw/carbon_tax_coverage.csv with columns: country, year (optional), coverage_pct")
    raise FileNotFoundError(f"Please download coverage data to {coverage_path}")

coverage_raw = pd.read_csv(coverage_path)
print(f"Coverage data shape: {coverage_raw.shape}")
print(coverage_raw.head(10))
print(f"\nCoverage pct range: {coverage_raw['coverage_pct'].min():.1f} – {coverage_raw['coverage_pct'].max():.1f}")

# Normalize to 0-1 if provided as 0-100
if coverage_raw['coverage_pct'].max() > 1:
    coverage_raw['coverage_pct'] = coverage_raw['coverage_pct'] / 100
    print("Normalized coverage_pct from 0-100 to 0-1 scale")

In [ ]:
# Align country names and merge
coverage_countries = set(coverage_raw['country'].unique())
df_countries = set(df['country'].unique())

unmatched = coverage_countries - df_countries
if unmatched:
    print(f"Coverage countries not in main df ({len(unmatched)}):")
    print(sorted(unmatched)[:20])

# Common name mismatches — extend as needed after inspecting unmatched
name_map = {
    'United States of America': 'United States',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'Korea, Republic of': 'South Korea',
    'Russian Federation': 'Russia',
}
coverage_raw['country'] = coverage_raw['country'].replace(name_map)

# Merge — panel merge if year column exists, cross-section otherwise
if 'year' in coverage_raw.columns:
    df_cov = df.merge(
        coverage_raw[['country', 'year', 'coverage_pct']],
        on=['country', 'year'], how='left'
    )
else:
    df_cov = df.merge(
        coverage_raw[['country', 'coverage_pct']].drop_duplicates('country'),
        on='country', how='left'
    )

# Untreated countries: coverage = 0
df_cov.loc[df_cov['post_carbon_tax'] == 0, 'coverage_pct'] = \
    df_cov.loc[df_cov['post_carbon_tax'] == 0, 'coverage_pct'].fillna(0)

coverage_match = df_cov.loc[df_cov['post_carbon_tax'] == 1, 'coverage_pct'].notna().mean()
print(f"\nCoverage match rate among treated obs: {coverage_match*100:.1f}%")
print(f"Treated obs with missing coverage: {df_cov.loc[df_cov['post_carbon_tax']==1, 'coverage_pct'].isna().sum()}")

In [ ]:
# Construct effective carbon price and run continuous treatment DiD
df_cov['effective_carbon_price'] = df_cov['tax_price'] * df_cov['coverage_pct']
df_cov['effective_carbon_price'] = df_cov['effective_carbon_price'].fillna(0)

print("Effective carbon price (treated only):")
print(df_cov.loc[df_cov['post_carbon_tax']==1, 'effective_carbon_price'].describe())

# DiD with effective carbon price
CONT_FORMULA = """
co2_per_capita_future_trend ~ effective_carbon_price
    + C(year) + C(country)
    + log_gdp + log_population + trade_openness
    + natural_resource_rents_per_gdp + fossil_pct_filled
"""

df_cov_clean = df_cov.dropna(subset=[
    'co2_per_capita_future_trend', 'effective_carbon_price',
    'log_gdp', 'log_population', 'trade_openness',
    'natural_resource_rents_per_gdp', 'fossil_pct_filled'
])

model_eff = smf.ols(CONT_FORMULA, data=df_cov_clean).fit(
    cov_type='cluster', cov_kwds={'groups': df_cov_clean['country']}
)

coef_eff = model_eff.params['effective_carbon_price']
pval_eff = model_eff.pvalues['effective_carbon_price']
print(f"\n=== Continuous Treatment (Effective Price = Price x Coverage) ===")
print(f"Coefficient: {coef_eff:.6f}  p-value: {pval_eff:.4f}")

# Comparison: tax_price alone
PRICE_FORMULA = CONT_FORMULA.replace('effective_carbon_price', 'tax_price')
model_price = smf.ols(PRICE_FORMULA, data=df_cov_clean).fit(
    cov_type='cluster', cov_kwds={'groups': df_cov_clean['country']}
)
coef_price = model_price.params['tax_price']
pval_price = model_price.pvalues['tax_price']
print(f"\n=== Continuous Treatment (Price Only, original) ===")
print(f"Coefficient: {coef_price:.6f}  p-value: {pval_price:.4f}")

## Coverage Rate Analysis — Effective Carbon Price

| Treatment Variable | Coefficient | p-value | Interpretation |
|--------------------|------------|---------|----------------|
| `tax_price` (original) | fill in | fill in | Price alone ignores coverage |
| `effective_carbon_price` | fill in | fill in | Price x coverage rate |

**Coverage data source:** fill in
**Coverage match rate:** fill in %

**Finding:** fill in after running